In [1]:
import glob
import os
import tqdm
import math

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
# import mpl_toolkits.axes_grid1
import japanize_matplotlib

import astropy
import astropy.io.fits
import astropy.units as u
import astroquery.vizier
from astropy.coordinates import SkyCoord
# from astropy.wcs import WCS
from spectral_cube import SpectralCube
import pylab

pylab.rcParams['font.family'] = 'serif'
pylab.rcParams['lines.linewidth'] = 0.5
matplotlib.rcParams["font.family"] = "serif"
matplotlib.rcParams["font.size"] = 15
pylab.rcParams["xtick.direction"] = "in"
pylab.rcParams["ytick.direction"] = "in"

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'Yu Gothic', 'Meirio', 'Takao', 'IPAexGothic', 'IPAPGothic', 'VL PGothic', 'Noto Sans CJK JP']

In [2]:
import sys
# sys.path.append('/home/elmegreen/galactic_bubble/photoutils/')
from processing import norm_res, normalize_rp, remove_nan, conv, data_view_rectangl, resize
from utils.ssd_model import nm_suppression

# sys.path.append('/home/elmegreen/jupyter/research/Bubble_Analysis/CO_SpitzerBubble/All_Bubble_Analysis/Analysis')
from Function_to_Detect_peak import find_verified_peak, _detect_and_characterize_peak, find_velocity_from_catalog, _check_signal_at_channel, gaussian_filter
from detect_rough_tools import extract_spectral, make_spitzer_fits, load_spitzer_fits, make_momont012_map

In [3]:
viz = astroquery.vizier.Vizier(columns=["*"])
viz.ROW_LIMIT = -1
MWP = viz.query_constraints(catalog="2019yCat..74881141J ")[0].to_pandas()

# ⬇️バブルの中に銀河中心を跨ぐものがあった場合、座標がバグる((359.9, 0.01)みたいに。それを(-0.01, 0.01)に変換して防ぐ)
MWP.loc[MWP["GLON"] >= 358.446500015535, "GLON"] -= 360
MWP.index = MWP['MWP'].to_list()

In [4]:
MWP

,MWP,GLON,GLAT,Disp,MajAxis,MinAxis,Reff,e_Reff,theta,e_theta,...,HR3,RelFlag,HierFlag,IDDR1,IDA14,Dist,IDCW,Simbad,_RA.icrs,_DE.icrs
2G0000794-0020817,2G0000794-0020817,0.0794,-0.2082,0.22,0.58,0.49,0.53,0.13,161,133,...,0.18,R,,1G000080-002075,G000.079-00.211,NaN,CN2,Simbad,266.6554,-28.9766
2G0001255+0003035,2G0001255+0003035,0.1255,0.0303,1.01,5.05,4.26,4.67,0.59,87,62,...,0.20,C,,1G000127+000485,G000.132+00.039,NaN,,Simbad,266.4501,-28.8133
2G0001301-0067518,2G0001301-0067518,0.1301,-0.6752,0.20,0.45,0.37,0.41,0.12,24,122,...,0.42,R,,,G000.129-00.674,NaN,CN4,Simbad,267.1427,-29.1749
2G0001400-0011719,2G0001400-0011719,0.1400,-0.1172,0.29,3.14,2.98,3.06,0.37,11,86,...,0.46,C,,1G000140-001173,G000.138-00.115,NaN,,Simbad,266.6025,-28.8776
2G0002791-0048490,2G0002791-0048490,0.2791,-0.4849,0.12,0.42,0.38,0.40,0.23,34,94,...,0.08,R,,,G000.279-00.482,NaN,CN7,Simbad,267.0442,-28.9491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2G3593514-0041492,2G3593514-0041492,-0.6486,-0.4149,0.34,0.44,0.35,0.40,0.24,52,83,...,0.08,C,,1G359350-004141,G359.349-00.417,NaN,,Simbad,266.4237,-29.7059
2G3596901+0006590,2G3596901+0006590,-0.3099,0.0659,0.10,0.59,0.51,0.55,0.06,77,79,...,0.13,C,,,G359.690+00.065,NaN,,Simbad,266.1557,-29.1661
2G3597425-0041121,2G3597425-0041121,-0.2575,-0.4112,0.94,1.97,1.82,1.90,0.68,27,106,...,0.44,R,,1G359737-004097,G359.740-00.412,NaN,CS2,Simbad,266.6538,-29.3700
2G3599224+0011336,2G3599224+0011336,-0.0776,0.1134,0.28,1.08,0.93,1.00,0.31,103,64,...,0.24,C,,,G359.930+00.108,NaN,,Simbad,266.2482,-28.9432


In [5]:
MWP.columns

Index(['MWP', 'GLON', 'GLAT', 'Disp', 'MajAxis', 'MinAxis', 'Reff', 'e_Reff',
       'theta', 'e_theta', 'Ecc', 'HR2', 'HR3', 'RelFlag', 'HierFlag', 'IDDR1',
       'IDA14', 'Dist', 'IDCW', 'Simbad', '_RA.icrs', '_DE.icrs'],
      dtype='object')

In [6]:
viz = astroquery.vizier.Vizier(columns=["*"])
viz.ROW_LIMIT = -1
bub_velocity_table = viz.query_constraints(catalog="J/MNRAS/438/426")[0].to_pandas()

# GLONを0-360度の範囲に正規化
bub_velocity_table['GLON'] = bub_velocity_table['GLON'] % 360
bub_velocity_table['GLON2'] = bub_velocity_table['GLON2'] % 360

print(f"読み込んだバブル速度カタログのエントリ数: {len(bub_velocity_table)}")
print("カタログの列名:", bub_velocity_table.columns.tolist())

読み込んだバブル速度カタログのエントリ数: 818
カタログの列名: ['MWP', 'GLON', 'GLAT', 'Reff', 'GLON2', 'GLAT2', 'Ref', 'VHII', 'D0', 'e_D0', 'r_D0', 'DK', 'e_DK', 'Mark', 'r_Mark', 'Simbad', '_RA.icrs', '_DE.icrs']


In [7]:
cygnus_bubble_catalogue = pd.read_csv('/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Bubble_Catalogue/cygnus_infer_catalogue.csv')
cygnus_bubble_catalogue['ra_center'] = (cygnus_bubble_catalogue['ra_min'] + cygnus_bubble_catalogue['ra_max'])/2
cygnus_bubble_catalogue['dec_center'] = (cygnus_bubble_catalogue['dec_min'] + cygnus_bubble_catalogue['dec_max'])/2
# all_bubble_catalogue = all_bubble_catalogue[2000:]
cygnus_bubble_catalogue

,Unnamed: 0,dec_min,ra_min,dec_max,ra_max,width_pix,height_pix,ra_center,dec_center
0,0,42.291897,308.650682,42.565254,308.999023,390.0,406.0,308.824853,42.428575
1,0,40.799006,307.460314,40.847080,307.523363,71.0,73.0,307.491838,40.823043
2,0,38.815480,308.056761,38.854625,308.107591,59.0,59.0,308.082176,38.835053
3,0,40.250445,308.097562,40.298526,308.160910,73.0,72.0,308.129236,40.274486
4,0,37.276328,306.827304,37.312209,306.873721,55.0,55.0,306.850512,37.294268
...,...,...,...,...,...,...,...,...,...
73,0,40.202114,305.540874,40.269707,305.628996,99.0,104.0,305.584935,40.235910
74,0,39.838105,309.394425,39.879458,309.447556,62.0,61.0,309.420990,39.858782
75,0,42.262325,309.294624,42.291281,309.332482,42.0,43.0,309.313553,42.276803
76,0,41.954736,309.778572,42.023760,309.871682,106.0,101.0,309.825127,41.989248


In [8]:
cygnus_path_list = sorted(
    glob.glob(
        '/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Cygnus-X/NRO45m/*12CO*.fits'))
print(len(cygnus_path_list))
print(cygnus_path_list[0])

1
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Cygnus-X/NRO45m/Cygnus_sp16_vs-40_ve040_dv0.25_12CO_Tmb.fits


In [9]:
zeroing_cygnus_path_list = sorted(
    glob.glob(
        '/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/processed_fits/Cygnus-X/*12CO*.fits'))
print(len(zeroing_cygnus_path_list))
print(zeroing_cygnus_path_list[0])

1
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/processed_fits/Cygnus-X/Cygnus_sp16_vs-40_ve040_dv0.25_12CO_Tmb_zeroing.fits


In [10]:
# 詳細な統計情報を追跡するための変数を初期化
velocity_stats = {
    'catalogue_vel_bubble': 0,
    'catalogue_vel_with_c18o': 0,  # 新規追加：カタログ速度使用かつC18Oピークあり
    'c18o_vel_bubble': 0,
    'co13_vel_bubble': 0,
    'no_vel_bubble': 0,
    'spitzer_error_bubble': 0,
    'total_bubble': 0
}

In [60]:
# 各バブルの詳細情報を記録するリスト
bubble_details = []

for each_path, zeroing_each_path in tqdm.tqdm(zip(cygnus_path_list, zeroing_cygnus_path_list)):
    # パスやディレクトリの設定
    base_dir = os.path.dirname(os.path.dirname(each_path))
    print(base_dir)
    region_name = each_path.split('/')[-1].split('+')[0]
    print(region_name)
    output_fig_dir = os.path.join('Spectral_fig', region_name)
    os.makedirs(output_fig_dir, exist_ok=True)

    Zeroing_12CO_fits_path = zeroing_each_path
    print(Zeroing_12CO_fits_path)
    Zeroing_13CO_fits_path = zeroing_each_path.replace("12CO","13CO")
    print(Zeroing_13CO_fits_path)
    Zeroing_C18O_fits_path = zeroing_each_path.replace("12CO","C18O")
    print(Zeroing_C18O_fits_path)

    Zeroing_cygnus_cube_fits_12CO = astropy.io.fits.open(Zeroing_12CO_fits_path)[0]
    Zeroing_cygnus_cube_fits_13CO = astropy.io.fits.open(Zeroing_13CO_fits_path)[0]
    Zeroing_cygnus_cube_fits_C18O = astropy.io.fits.open(Zeroing_C18O_fits_path)[0]

    # FITSファイルのパスを構築
    _12CO_fits_path = each_path
    _13CO_fits_path = each_path.replace("12CO", "13CO")
    C18O_fits_path = each_path.replace("12CO", "C18O")
    print(_12CO_fits_path, _13CO_fits_path, C18O_fits_path)

    # FITSファイルを開く
    cygnus_cube_fits_12CO = astropy.io.fits.open(_12CO_fits_path)[0]
    cygnus_cube_fits_13CO = astropy.io.fits.open(_13CO_fits_path)[0]
    cygnus_cube_fits_C18O = astropy.io.fits.open(C18O_fits_path)[0]

    # WCSと速度軸の情報を抽出
    w_co = astropy.wcs.WCS(cygnus_cube_fits_12CO.header)
    cube = SpectralCube.read(cygnus_cube_fits_12CO)
    vaxis = cube.spectral_axis.to_value(u.km/u.s)
    dv = abs(cygnus_cube_fits_12CO.header['CDELT3']) / 1000.0

    # カタログフィルタリング
    ny, nx = cygnus_cube_fits_12CO.data.shape[1:3]
    glon_min, glat_min, _ = w_co.all_pix2world(nx, 0, 0, 0)
    glon_max, glat_max, _ = w_co.all_pix2world(0, ny, 0, 0)
    print(glon_min, glon_max, glat_min, glat_max)

    catalog_glon_min, catalog_glat_min, _ = w_co.all_pix2world(0, nx, 0, 0)
    catalog_glon_max, catalog_glat_max, _ = w_co.all_pix2world(ny, 0, 0, 0)

    # 2. 銀河座標(Frame='galactic')を赤道座標(Frame='icrs' or 'fk5')に変換
    # 最小値側の変換
    coords_min = SkyCoord(l=catalog_glon_min*u.degree, b=catalog_glat_min*u.degree, frame='galactic')
    ra_min = coords_min.icrs.ra.degree
    dec_min = coords_min.icrs.dec.degree

    # 最大値側の変換
    coords_max = SkyCoord(l=catalog_glon_max*u.degree, b=catalog_glat_max*u.degree, frame='galactic')
    ra_max = coords_max.icrs.ra.degree
    dec_max = coords_max.icrs.dec.degree

    # 3. 変換後のRA, Decを使ってクエリを実行
    # ※範囲指定(min/max)の大小関係が逆転する場合があるため、念のためmin()とmax()をとると安全です
    ra_low, ra_high = sorted([ra_min, ra_max])
    dec_low, dec_high = sorted([dec_min, dec_max])
    print(ra_low, ra_high, dec_low, dec_high)

    cygnus_bubble_catalogue_selected = cygnus_bubble_catalogue.query(
        # f"{ra_low} <= ra_center <= {ra_high} and {dec_low} <= dec_center <= {dec_high}"
        f"{ra_low} <= ra_center <= {ra_high}"
    ).reset_index()
    MWP_selected = MWP.query(f"{glon_min} <= GLON <= {glon_max} and {glat_min} <= GLAT <= {glat_max}"
    ).reset_index()
    # cygnus_bubble_catalogue_selected = cygnus_bubble_catalogue.query(
    #     f"{glon_min} <= ra_center <= {glon_max} and {glat_min} <= dec_center <= {glat_max}"
    # ).reset_index()

1it [00:00, 60.30it/s]

/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Cygnus-X
Cygnus_sp16_vs-40_ve040_dv0.25_12CO_Tmb.fits
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/processed_fits/Cygnus-X/Cygnus_sp16_vs-40_ve040_dv0.25_12CO_Tmb_zeroing.fits
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/processed_fits/Cygnus-X/Cygnus_sp16_vs-40_ve040_dv0.25_13CO_Tmb_zeroing.fits
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/processed_fits/Cygnus-X/Cygnus_sp16_vs-40_ve040_dv0.25_C18O_Tmb_zeroing.fits
/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Cygnus-X/NRO45m/Cygnus_sp16_vs-40_ve040_dv0.25_12CO_Tmb.fits /home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Cygnus-X/NRO45m/Cygnus_sp16_vs-40_ve040_dv0.25_13CO_Tmb.fits /home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Cygnus-X/NRO45m/Cygnus_sp16_vs-40_ve040_dv0.25_C18O_Tmb.fits
77.14803676100189 82.40006209732758 -1.5666666666639142 1.752083333330776
306.8554830256363 309.870551535

In [64]:
cygnus_cube_fits_12CO.data.shape

(320, 1593, 2520)

In [67]:
MWP_selected

,index,MWP,GLON,GLAT,Disp,MajAxis,MinAxis,Reff,e_Reff,theta,...,HR3,RelFlag,HierFlag,IDDR1,IDA14,Dist,IDCW,Simbad,_RA.icrs,_DE.icrs
0,2G0774273-0001360,2G0774273-0001360,77.4274,-0.0136,0.39,1.00,0.80,0.90,0.30,31,...,0.24,R,,,G077.431-00.015,NaN,,Simbad,307.0071,38.5826
1,2G0774339+0167929,2G0774339+0167929,77.4339,1.6793,0.55,2.29,1.70,2.01,0.38,175,...,0.28,R,,,G077.421+01.685,1.5,,Simbad,305.2287,39.5624
2,2G0776050+0055412,2G0776050+0055412,77.6050,0.5541,0.30,0.70,0.51,0.61,0.30,174,...,0.09,C,,,G077.608+00.554,NaN,,Simbad,306.5468,39.0572
3,2G0778938+0167673,2G0778938+0167673,77.8938,1.6767,0.65,2.52,2.03,2.29,0.40,174,...,0.12,C,,,G077.926+01.678,1.5,,Simbad,305.5717,39.9389
4,2G0779686+0000279,2G0779686+0000279,77.9686,0.0028,0.75,2.08,1.52,1.82,0.68,18,...,0.23,R,,,G077.977-00.004,1.5,,Simbad,307.3965,39.0311
5,2G0780301+0061158,2G0780301+0061158,78.0301,0.6116,0.61,3.10,2.30,2.73,0.55,2,...,0.28,R,,,G078.032+00.606,1.5,,Simbad,306.8050,39.4367
6,2G0781504-0054972,2G0781504-0054972,78.1504,-0.5497,0.28,2.46,2.02,2.25,0.26,68,...,0.17,R,,,G078.174-00.550,1.5,,Simbad,308.1079,38.8518
7,2G0781849-0032307,2G0781849-0032307,78.1850,-0.3231,1.02,5.65,4.46,5.09,0.81,38,...,0.30,R,,,G078.177-00.363,1.5,,Simbad,307.8995,39.0139
8,2G0784466+0109013,2G0784466+0109013,78.4466,1.0901,1.63,5.01,2.89,4.09,1.25,100,...,0.24,R,,,,NaN,,Simbad,306.6130,40.0536
9,2G0786924+0035385,2G0786924+0035385,78.6924,0.3538,0.58,1.19,0.81,1.02,0.32,166,...,0.39,R,,,G078.689+00.354,1.5,,Simbad,307.5796,39.8229


In [68]:
cygnus_bubble_catalogue_selected

,index,Unnamed: 0,dec_min,ra_min,dec_max,ra_max,width_pix,height_pix,ra_center,dec_center
0,0,0,42.291897,308.650682,42.565254,308.999023,390.0,406.0,308.824853,42.428575
1,1,0,40.799006,307.460314,40.847080,307.523363,71.0,73.0,307.491838,40.823043
2,2,0,38.815480,308.056761,38.854625,308.107591,59.0,59.0,308.082176,38.835053
3,3,0,40.250445,308.097562,40.298526,308.160910,73.0,72.0,308.129236,40.274486
4,5,0,38.827698,308.070601,38.896838,308.155209,99.0,104.0,308.112905,38.862268
5,7,0,38.937297,307.903431,39.005556,307.989512,100.0,102.0,307.946472,38.971426
6,9,0,42.253673,309.339162,42.280251,309.373672,38.0,39.0,309.356417,42.266962
7,12,0,40.829334,307.015700,40.900314,307.109247,105.0,107.0,307.062473,40.864824
8,13,0,42.082235,309.584376,42.127088,309.642958,66.0,66.0,309.613667,42.104662
9,17,0,39.863763,308.728812,40.000478,308.903299,203.0,204.0,308.816056,39.932121


In [70]:
print("Catalog 1 columns:", MWP.columns.tolist())
print("Catalog 2 columns:", cygnus_bubble_catalogue_selected.columns.tolist())

Catalog 1 columns: ['MWP', 'GLON', 'GLAT', 'Disp', 'MajAxis', 'MinAxis', 'Reff', 'e_Reff', 'theta', 'e_theta', 'Ecc', 'HR2', 'HR3', 'RelFlag', 'HierFlag', 'IDDR1', 'IDA14', 'Dist', 'IDCW', 'Simbad', '_RA.icrs', '_DE.icrs']
Catalog 2 columns: ['index', 'Unnamed: 0', 'dec_min', 'ra_min', 'dec_max', 'ra_max', 'width_pix', 'height_pix', 'ra_center', 'dec_center']


In [71]:
import pandas as pd

# 1. Catalog 2 から不要な列を削除
# 'index' と 'Unnamed: 0' をドロップします
catalog2_clean = cygnus_bubble_catalogue_selected.drop(columns=['index', 'Unnamed: 0'], errors='ignore')

# 2. Catalog 1 の列名を Catalog 2 の形式にマッピング（リネーム）
# 結合のキーとなる RA/Dec を揃えます
rename_dict = {
    '_RA.icrs': 'ra_center',
    '_DE.icrs': 'dec_center'
}
catalog1_renamed = MWP_selected.rename(columns=rename_dict)

# 3. 結合 (Merge)
# ra_center と dec_center をキーにして結合します。
# 浮動小数点の微小な誤差で結合できないのを防ぐため、必要に応じて round(6) などを検討してください。
merged_df = pd.merge(catalog1_renamed, catalog2_clean, on=['ra_center', 'dec_center'], how='inner')

# 4. 列の並び替え (Catalog 2 の列を先頭に持ってくる)
# Catalog 2 にあった列名をリストアップ
c2_cols = ['dec_min', 'ra_min', 'dec_max', 'ra_max', 'width_pix', 'height_pix', 'ra_center', 'dec_center']
# それ以外の Catalog 1 由来の列を取得
other_cols = [c for c in merged_df.columns if c not in c2_cols]

# 最終的な DataFrame
final_catalogue = merged_df[c2_cols + other_cols]

# 結果の確認
print(f"結合後のエントリ数: {len(final_catalogue)}")
print(final_catalogue.head())

結合後のエントリ数: 0
Empty DataFrame
Columns: [dec_min, ra_min, dec_max, ra_max, width_pix, height_pix, ra_center, dec_center, index, MWP, GLON, GLAT, Disp, MajAxis, MinAxis, Reff, e_Reff, theta, e_theta, Ecc, HR2, HR3, RelFlag, HierFlag, IDDR1, IDA14, Dist, IDCW, Simbad]
Index: []

[0 rows x 29 columns]


In [74]:
# --- レイアウト設定 ---
bubbles_per_row = 2
rows_per_page = 100  # 1ページに表示する行数
bubbles_per_page = bubbles_per_row * rows_per_page
total_bubbles = len(cygnus_bubble_catalogue_selected) + len(MWP_selected)
num_pages = math.ceil(total_bubbles / bubbles_per_page)
    
# キャッシュ初期化
spitzer_path_cached = None

# ページごとのループ
for page in range(num_pages):
    start_idx = page * bubbles_per_page
    end_idx = min(start_idx + bubbles_per_page, total_bubbles)
    current_page_count = end_idx - start_idx

    # ページ内の必要行数を計算
    current_rows_in_page = math.ceil(current_page_count / bubbles_per_row)

    # ページ全体のFigure作成
    fig = plt.figure(figsize=(15*bubbles_per_row, 8 * current_rows_in_page))

    # サブフィギュアの作成
    subfigs_obj = fig.subfigures(current_rows_in_page, bubbles_per_row, wspace=0.1, hspace=0.1)

    # 【修正ポイント】戻り値が配列なら平坦化し、単一オブジェクトならリストに包む
    if isinstance(subfigs_obj, np.ndarray):
        subfigs = subfigs_obj.flatten()
    else:
        subfigs = [subfigs_obj]

    for p_idx, subfig in enumerate(subfigs):
        i_in_catalogue = start_idx + p_idx
        if i_in_catalogue >= total_bubbles:
            subfig.set_visible(False)
            continue

        each_catalogue = cygnus_bubble_catalogue.iloc[i_in_catalogue]
        i = each_catalogue.name # 元のインデックス
        
        # --- データ処理ロジック (省略なし) ---
        velocity_stats['total_bubble'] += 1

        bubble_info = {
            'region': region_name,
            'catalogue_index': i,
            'ra_center': each_catalogue['ra_center'],
            'dec_center': each_catalogue['dec_center'],
            'velocity_source': None,
            'v_peak': None,
            'fwhm_vel': None,
            'has_c18o_peak': False,
            'status': None
        }

        # Spitzerデータのロード（キャッシュを利用）
        current_spitzer_path = os.path.join("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Cygnus-X/Spitzer/processed_fits/RGB/")
        if current_spitzer_path != spitzer_path_cached:
            spitzer_rfits, spitzer_gfits, spitzer_data, w_spitzer = load_spitzer_fits(current_spitzer_path)
            spitzer_path_cached = current_spitzer_path

        # スペクトルデータを抽出
        cut_data_12CO = extract_spectral(w_co, cygnus_cube_fits_12CO, each_catalogue)
        cut_data_13CO = extract_spectral(w_co, cygnus_cube_fits_13CO, each_catalogue)
        cut_data_C18O = extract_spectral(w_co, cygnus_cube_fits_C18O, each_catalogue)

        zeroing_cut_data_12CO = extract_spectral(w_co, Zeroing_cygnus_cube_fits_12CO, each_catalogue)
        zeroing_cut_data_13CO = extract_spectral(w_co, Zeroing_cygnus_cube_fits_13CO, each_catalogue)
        zeroing_cut_data_C18O = extract_spectral(w_co, Zeroing_cygnus_cube_fits_C18O, each_catalogue)

        # 平均スペクトルを計算
        mean_data_12CO = np.nanmean(cut_data_12CO, axis=(1, 2))
        mean_data_13CO = np.nanmean(cut_data_13CO, axis=(1, 2))
        mean_data_C18O = np.nanmean(cut_data_C18O, axis=(1, 2))

        # C18Oのピーク検証
        c18o_v_peak, c18o_t_peak, c18o_fwhm_vel, c18o_peak_channel = find_verified_peak(
            c18o_spec=mean_data_C18O,
            co12_spec=mean_data_12CO,
            co13_spec=mean_data_13CO,
            vaxis=vaxis, dv=dv
        )

        bubble_info['has_c18o_peak'] = c18o_v_peak is not None

        # [cite_start]カタログ（Beaumont & Williams 2014等）から速度情報を検索
        catalog_v_peak, catalog_fwhm_vel, catalog_info = find_velocity_from_catalog(
            each_catalogue['ra_center'], 
            each_catalogue['dec_center'], 
            bub_velocity_table,
            search_radius=(each_catalogue['dec_max'] - each_catalogue['dec_min'])/2
        )

        if catalog_v_peak is not None:
            used_tracer = f"Catalog (Row {catalog_info['catalog_index']})"
            v_peak = catalog_v_peak
            fwhm_vel = catalog_fwhm_vel
            t_peak = np.nan
            peak_channel = np.argmin(np.abs(vaxis - v_peak))
            
            if bubble_info['has_c18o_peak'] and abs(c18o_v_peak - catalog_v_peak) <= 10.0:
                velocity_stats['catalogue_vel_with_c18o'] += 1
            else:
                bubble_info['has_c18o_peak'] = False 
            
            velocity_stats['catalogue_vel_bubble'] += 1
            bubble_info.update({'velocity_source': 'Catalog', 'v_peak': v_peak, 'fwhm_vel': fwhm_vel, 'status': 'Success'})
            
        elif c18o_v_peak is not None:
            used_tracer = "C18O (verified)"
            v_peak, t_peak, fwhm_vel, peak_channel = c18o_v_peak, c18o_t_peak, c18o_fwhm_vel, c18o_peak_channel
            velocity_stats['c18o_vel_bubble'] += 1
            bubble_info.update({'velocity_source': 'C18O', 'v_peak': v_peak, 'fwhm_vel': fwhm_vel, 'status': 'Success'})
            
        else:
            used_tracer = "13CO"
            v_peak, t_peak, fwhm_vel, peak_channel = _detect_and_characterize_peak(
                mean_data_13CO, vaxis, dv, height_factor=5.0, prominence_factor=3.0
            )
            if v_peak is not None:
                velocity_stats['co13_vel_bubble'] += 1
                bubble_info.update({'velocity_source': '13CO', 'v_peak': v_peak, 'fwhm_vel': fwhm_vel, 'status': 'Success'})
            else:
                velocity_stats['no_vel_bubble'] += 1
                bubble_info.update({'velocity_source': 'None', 'status': 'No velocity detected'})
                subfig.text(0.5, 0.5, f"No velocity detected\nIdx: {i}", ha='center', va='center', fontsize=12)
                bubble_details.append(bubble_info)
                continue

        # 速度範囲の設定
        range_start_vel = v_peak - fwhm_vel * 2.5
        range_end_vel = v_peak + fwhm_vel * 2.5
        range_indices = np.where((vaxis >= range_start_vel) & (vaxis <= range_end_vel))[0]

        # Momentマップ作成 (FUGINデータを使用)
        datadict_12CO = make_momont012_map(cygnus_cube_fits_12CO, w_co, zeroing_cut_data_12CO[range_indices], vaxis[range_indices], v_center=v_peak)
        datadict_13CO = make_momont012_map(cygnus_cube_fits_13CO, astropy.wcs.WCS(cygnus_cube_fits_13CO.header), zeroing_cut_data_13CO[range_indices], vaxis[range_indices], v_center=v_peak)
        datadict_C18O = make_momont012_map(cygnus_cube_fits_C18O, astropy.wcs.WCS(cygnus_cube_fits_C18O.header), zeroing_cut_data_C18O[range_indices], vaxis[range_indices], v_center=v_peak)

        # Spitzer画像の準備
        new_hdu_list_r, new_hdu_list_g = make_spitzer_fits(spitzer_rfits, spitzer_gfits, w_spitzer, spitzer_data, each_catalogue)
        if new_hdu_list_r == 0:
            velocity_stats['spitzer_error_bubble'] += 1
            bubble_info['status'] = 'Spitzer error'
            subfig.text(0.5, 0.5, f"Spitzer Error\nIdx: {i}", ha='center', va='center', fontsize=12)
            bubble_details.append(bubble_info)
            continue
            
        cut_spitzer = np.concatenate([
            new_hdu_list_r[0].data[:,:,None],
            new_hdu_list_g[0].data[:,:,None],
            np.zeros(new_hdu_list_r[0].data.shape)[:,:,None]
        ], axis=2)

        # --- プロット処理 ---
        v_peak_str = f"{v_peak:.1f}" if v_peak is not None else "N/A"
        c18o_v_peak_str = f"{c18o_v_peak:.1f}" if c18o_v_peak is not None else "N/A"

        if "Catalog" in used_tracer:
            title = (f"Peak from Hou et al. 2013, "
                     f"V_HII={v_peak_str} km/s, C18O peak={c18o_v_peak_str} km/s")
        else:
            title = (f"Catalogue: {i}, Bubble: {each_catalogue.get('name', 'N/A')}, "
                     f"Peak from {used_tracer} at V_lsr={v_peak_str} km/s")

        # subfigにタイトルを設定
        subfig.suptitle(title, fontsize=20, fontweight='bold', y=0.98)
        
        gs = GridSpec(3, 7, figure=subfig, width_ratios=[1, 1, 1, 1, 1, 1, 2], 
                      wspace=0.8, hspace=0.3, left=0.05, right=0.95, top=0.90, bottom=0.08)

        # 1. Spitzer赤外線画像
        ax = subfig.add_subplot(gs[:3, :3], projection=astropy.wcs.WCS(new_hdu_list_r[0].header))
        ax.imshow(cut_spitzer)
        ax.tick_params(labelsize=8)
        ax.set_xlabel('Galactic Longitude', fontsize=15)
        ax.set_ylabel('Galactic Latitude', fontsize=15)

        # 2. スペクトル (12CO, 13CO, C18O)
        spectral_data_list = [("12CO", mean_data_12CO), ("13CO", mean_data_13CO), ("C18O", mean_data_C18O)]
        for idx, (label, spec_data) in enumerate(spectral_data_list):
            ax = subfig.add_subplot(gs[idx, 3:6])
            ax.plot(vaxis, spec_data, "k", label=label, drawstyle='steps-mid')
            if len(range_indices) > 0:
                ax.plot(vaxis[range_indices], spec_data[range_indices], "o", color='red', markersize=2)
                ax.axvspan(range_start_vel, range_end_vel, alpha=0.2, color='red')
    
            ax.axvline(v_peak, color='blue', ls='--', lw=1, label=f'Peak: {v_peak_str} km/s')
            if bubble_info['has_c18o_peak'] and idx == 2:
                ax.axvline(c18o_v_peak, color='green', ls=':', lw=2, label=f'C18O peak: {c18o_v_peak_str} km/s')
    
            ax.set_xlim([-80, 80])
            ax.tick_params(axis='both', labelsize=12)
            ax.set_title(label, fontsize=15)
            ax.set_ylabel("Mean Tmb [K]", fontsize=12)
            ax.legend(fontsize=10, loc='upper right')

        # 3. 積分強度マップ (Moment 0)
        moms = [datadict_12CO, datadict_13CO, datadict_C18O]
        for idx, mdata in enumerate(moms):
            ax = subfig.add_subplot(gs[idx, 6:7])
            mdata = mdata['moment0']
            ax.imshow(mdata, origin='lower', interpolation='none', cmap='viridis')

            # コントア作成
            x_width = mdata.shape[1]
            y_width = mdata.shape[0]
            x = np.linspace(0, x_width, x_width)
            y = np.linspace(0, y_width, y_width)
            X, Y = np.meshgrid(x, y)
            sig1 = 1 / (2 * (np.log(2)) ** 0.5)

            # 修正：resize(conv(...)) の結果が正しく渡るように
            try:
                spitzer_contour_data = resize(conv(int(x_width), sig1, cut_spitzer), (int(y_width), int(x_width)))[:,:,1]
                contour = ax.contour(X, Y, spitzer_contour_data, levels=[0.1, 0.3, 0.5], colors=['w'], linewidths=3)
                ax.clabel(contour, inline=True, fontsize=12)
            except Exception as e:
                print(f"Contour error at index {i}: {e}")

            r_pix = x_width/4
            center_pix = x_width/2
            tick_pos = center_pix + np.array([-1, 0, 1]) * r_pix
            ax.set_xticks(tick_pos); ax.set_xticklabels(['-R', '0', 'R'])
            ax.set_yticks(tick_pos); ax.set_yticklabels(['-R', '0', 'R'])
            ax.set_title(['12CO', '13CO', 'C18O'][idx] + ' Moment 0', fontsize=15)

        bubble_details.append(bubble_info)

    # ページ保存
    # dir_name = f"FUGIN_Bubble_Profile/{region_name}"
    # os.makedirs(dir_name, exist_ok=True)

    out_name = f"Cygnus-X_Bubble_Profile/Summary_Cygnus-X.png"

    plt.savefig(out_name, dpi=72, bbox_inches='tight')
    # plt.close(fig)
    plt.show()
    print(f"Generated: {out_name}")

<Figure size 3000x24800 with 0 Axes>

Generated: Cygnus-X_Bubble_Profile/Summary_Cygnus-X.png
